## 1. Defining Game Quality

In [ ]:
# Read scv
import pandas as pd
df = pd.read_csv("data\steam_games_dataset.csv")
df.head()

In [ ]:
# Caculate the positive ratio for each games
df["positive_ratio"] = df["positive"]/(df["positive"]+df["negative"])
df[["name","positive","negative","positive_ratio"]].head()

In [ ]:
# Naive ranking based only on positive ratio
df_filtered = df[df["positive"]+df["negative"] > 1000]
df_filtered.sort_values(by="positive_ratio",ascending=False)[["name","positive_ratio"]].head()

In [ ]:
# Bayesian-adjusted ranking
C = df["positive_ratio"].mean()
m = 10000
df["bayesian_score"] = ((df["positive"]+C*m)/(df["positive"]+df["negative"]+m))
df.sort_values(by="bayesian_score",ascending=False)[["name","positive_ratio"]].head()

#### Insight: A better way to identify the best Steam games

Simply ranking games by positive review ratio can be misleading, because games with very few reviews may appear at the top. To address this, I used a Bayesian-style score that combines review ratio with review volume. This produces a more reliable ranking and highlights games that are both highly rated and widely reviewed.

## 2. Free vs Paid Games

In [ ]:
df["game_type"] = df["price"].apply(lambda x: "Free" if x == 0 else "Paid")
df[["name", "price", "game_type"]].head()

In [ ]:
df["game_type"].value_counts()
df.groupby("game_type")[["positive_ratio", "bayesian_score"]].mean()
df_paid_free_filtered = df[df["positive"] + df["negative"] > 1000]
df_paid_free_filtered.groupby("game_type")[["positive_ratio", "bayesian_score"]].mean()
df_paid_free_filtered.groupby("game_type")[["positive_ratio", "bayesian_score"]].median()
summary_free_paid = df_paid_free_filtered.groupby("game_type").agg(
    game_count=("name", "count"),
    avg_positive_ratio=("positive_ratio", "mean"),
    median_positive_ratio=("positive_ratio", "median"),
    avg_bayesian_score=("bayesian_score", "mean"),
    median_bayesian_score=("bayesian_score", "median")
)

summary_free_paid

Whether a game is free or paid does not fully determine its review performance, but paid games show consistently higher average and median ratings, suggesting that paid titles are more likely to receive stronger user approval overall.

## 3. Quality vs Popularity

In [ ]:
top_rated = df.sort_values(by="bayesian_score", ascending=False)[
    ["name", "bayesian_score", "ccu", "price", "average_forever"]
].head(10)

top_popular = df.sort_values(by="ccu", ascending=False)[
    ["name", "ccu", "bayesian_score", "price", "average_forever"]
].head(10)

top_rated, top_popular

In [ ]:
def ccu_bucket(x):
    if x <= 1000:
        return "Low"
    elif x <= 10000:
        return "Medium"
    else:
        return "High"

df_ccu = df[df["ccu"] > 0].copy()
df_ccu["ccu_bucket"] = df_ccu["ccu"].apply(ccu_bucket)

ccu_summary = df_ccu.groupby("ccu_bucket").agg(
    game_count=("name", "count"),
    avg_positive_ratio=("positive_ratio", "mean"),
    median_positive_ratio=("positive_ratio", "median"),
    avg_bayesian_score=("bayesian_score", "mean"),
    median_bayesian_score=("bayesian_score", "median")
)

ccu_summary

### Insight: Quality and Popularity Are Positively Related

Games with higher CCU generally achieve higher review scores, suggesting a positive relationship between popularity and perceived quality. However, the improvement in Bayesian score is much more pronounced from the low-CCU group to the medium-CCU group, while the difference between medium- and high-CCU games is relatively small. This suggests that popularity and quality are related, but extremely high popularity does not necessarily imply substantially better ratings.